# Integrated CO2-aroma model selection and DOE

This notebook evaluates a formal model candidate where the gas-liquid partition model includes sugar:

$$K_i(T,E,G,F)=\frac{\gamma_i^{UNIFAC}(T,x_{water},x_{ethanol},x_{G+F})P_i^{sat,Antoine}(T)}{RTC_{tot,L}}.$$

The aroma loss is driven by an effective gas stripping flow:

$$r_{loss,i}=\alpha_iK_iq_{gas}C_{L,i}.$$

Three alternatives are compared:

- `instant`: \(q_{gas}=q_{CO2,prod}\)
- `threshold`: \(q_{gas}=\max(q_{CO2,prod}-q_0,0)\)
- `lag_threshold`: \(\tau dq_{gas}/dt=\max(q_{CO2,prod}-q_0,0)-q_{gas}\)

The selected model is then used to compute current-data FIM diagnostics and a natural-must DOE campaign.


In [1]:
from pathlib import Path
import json
import pandas as pd
RESULTS = Path('results/integrated_co2_aroma_doe')
print(RESULTS.resolve())


C:\Users\ctorrealba\OneDrive - Viña Concha y Toro S.A\Documentos\Doctorado\Artículos\Artículo - Estimación_dFBA\pyomo-doe\fermentation_model\pilot_2025\results\integrated_co2_aroma_doe


## Integrated Model Selection

In [2]:
pd.read_csv(RESULTS / 'integrated_model_selection_summary.csv')

,mode,data_wsse,n_data_residuals,n_parameters,active_bound_count,aicc,bic,selection_score,description,selected
0,threshold,5541.601829,674,12,2,5566.073841,5619.760590,5659.760590,Gas flow activates only above a fraction of th...,True
1,lag_threshold,5627.179878,674,13,2,5653.731393,5711.851870,5751.851870,Thresholded production-rate transform followed...,False
2,instant,9183.543223,674,11,2,9205.942014,9255.188754,9295.188754,Current model: stripping gas flow is directly ...,False


Selected CO2 mode: `threshold`.

In [3]:
pd.read_csv(RESULTS / 'theta_selected_integrated_model.csv', index_col=0).head(80)

,0
mu0,0.068971
sN,8.751446
qN,0.015772
qXG,0.081491
qXF,0.070933
betaG0,1.186336
sG,0.097455
betaF0,0.314510
sF,0.178295
qEG,1.112514


## Fit Metrics

In [4]:
pd.read_csv(RESULTS / 'integrated_aroma_pool_metrics.csv')

,species,pool,n,rmse,mae,bias,median_obs,median_pred,relative_rmse_to_median,relative_bias_to_median,co2_mode,partition_mode,mode,aroma_wsse,aroma_n_residuals
0,ethyl_acetate,condensate,17,0.080622,0.072791,0.072791,0.012473,0.084332,6.463819,5.836007,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
1,ethyl_acetate,retained,17,4.338076,3.403935,-3.052180,8.293908,5.652266,0.523044,-0.368003,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
2,ethyl_acetate,total,52,3.240500,2.203117,-1.840234,7.589931,4.822334,0.426947,-0.242457,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
3,ethyl_octanoate,condensate,44,0.009973,0.005924,-0.003614,0.012386,0.010312,0.805137,-0.291752,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
4,ethyl_octanoate,retained,44,0.022151,0.015530,-0.005946,0.034524,0.035839,0.641608,-0.172215,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
5,ethyl_octanoate,total,52,0.025124,0.016414,-0.008089,0.044668,0.043495,0.562462,-0.181083,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
6,isoamyl_acetate,condensate,44,0.565329,0.320249,-0.257803,0.479512,0.319312,1.178967,-0.537636,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
7,isoamyl_acetate,retained,44,1.919644,1.398745,-0.756859,5.203077,4.908258,0.368944,-0.145464,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
8,isoamyl_acetate,total,52,2.115103,1.411503,-0.858560,5.258392,4.508426,0.402234,-0.163274,instant,water_ethanol_GF_as_glucose,instant,8925.015212,366
9,ethyl_acetate,condensate,17,0.048799,0.040312,0.037203,0.012473,0.054231,3.912456,2.982693,threshold,water_ethanol_GF_as_glucose,threshold,5282.593048,366


In [5]:
pd.read_csv(RESULTS / 'integrated_co2_metrics.csv')

,mode,batch,n,scale_factor,rmse,mae,bias,relative_rmse,relative_bias,corr
0,instant,25170,191,1.939520,0.383478,0.333546,-0.173784,1.242877,-0.563244,0.328081
1,instant,25171,117,4.026690,0.174973,0.140931,0.054382,0.391057,0.121542,0.959637
2,threshold,25170,191,1.977125,0.387984,0.337168,-0.179972,1.257478,-0.583302,0.316926
3,threshold,25171,117,7.753600,0.159516,0.122806,-0.075800,0.356510,-0.169409,0.970796
4,lag_threshold,25170,191,2.030760,0.379447,0.330970,-0.171902,1.229809,-0.557144,0.350363
5,lag_threshold,25171,117,7.759265,0.159310,0.119424,-0.071167,0.356050,-0.159055,0.968947


## FIM Diagnostics

The FIM uses finite differences in log-parameter space for positive parameters and direct finite differences for the bounded threshold fraction.


In [6]:
json.load(open(RESULTS / 'target_parameters_integrated_selected.json'))

['mu0',
 'qN',
 'betaG0',
 'betaF0',
 'qEG',
 'qEF',
 'iG',
 'iE',
 'Kd0',
 'gammaG0',
 'gammaF0',
 'kPyrS_N',
 'kPyrS_stat',
 'kPyrO2',
 'kPyrDrain',
 'kAldPyr',
 'kAldS_N',
 'kAldS_stat',
 'kAldO2',
 'kAldRed',
 'kAcAld',
 'kAcStress',
 'kAcAssim',
 'k_EA_growth',
 'k_EA_stationary',
 'k_IAA_growth',
 'k_IAA_stationary',
 'k_EO_growth',
 'k_EO_stationary',
 'alpha_EA_loss',
 'alpha_IAA_loss',
 'alpha_EO_loss',
 'k_EA_XE',
 'k_EA_XE_Nlim',
 'qgas_threshold_fraction']

In [7]:
pd.read_csv(RESULTS / 'parameter_estimability_integrated_current.csv')

,analysis,parameter,theta,std_approx,approx_95_multiplier,fim_diag,active_bound,classification
0,integrated_current,mu0,0.068971,0.011540,1.022877e+00,36444.096657,False,well_estimated
1,integrated_current,qN,0.015772,0.014968,1.029772e+00,25801.997192,False,well_estimated
2,integrated_current,betaG0,1.186336,0.036653,1.074484e+00,49049.661775,False,well_estimated
3,integrated_current,betaF0,0.314510,0.203568,1.490322e+00,3797.400692,False,well_estimated
4,integrated_current,qEG,1.112514,0.037054,1.075328e+00,51100.240276,False,well_estimated
5,integrated_current,qEF,1.278010,0.096865,1.209075e+00,17503.251115,False,well_estimated
6,integrated_current,iG,0.013971,0.202278,1.486559e+00,1603.251858,False,well_estimated
7,integrated_current,iE,0.015660,0.082558,1.175641e+00,7330.089615,False,well_estimated
8,integrated_current,Kd0,0.000656,0.083077,1.176838e+00,202.598902,False,well_estimated
9,integrated_current,gammaG0,0.070310,0.093061,1.200094e+00,8712.441757,False,well_estimated


In [8]:
pd.read_csv(RESULTS / 'weak_directions_integrated_current.csv')

,analysis,weak_direction,eigenvalue,dominant_parameters,dominant_abs_loadings
0,integrated_current,1,-1.275997e-14,"kAcAssim, kAcStress, k_EA_XE, kAldRed, gammaF0...","0.764, 0.646, 0.000, 0.000, 0.000, 0.000, 0.00..."
1,integrated_current,2,2.849137e-15,"kAcStress, kAcAssim, k_EA_XE, kAldRed, gammaF0...","0.764, 0.646, 0.000, 0.000, 0.000, 0.000, 0.00..."
2,integrated_current,3,1.389942e-08,"k_EA_XE, k_EA_XE_Nlim, k_EA_growth, kAldRed, g...","1.000, 0.001, 0.000, 0.000, 0.000, 0.000, 0.00..."
3,integrated_current,4,8.469289e-04,"kAldRed, kAldPyr, kPyrS_stat, kAldO2, kAldS_st...","1.000, 0.012, 0.006, 0.005, 0.003, 0.002, 0.00..."
4,integrated_current,5,3.628691e-02,"kPyrS_stat, kPyrO2, kPyrDrain, kAldS_stat, kAl...","0.997, 0.070, 0.009, 0.008, 0.006, 0.006, 0.00..."
5,integrated_current,6,2.485174e-01,"gammaF0, gammaG0, betaF0, iG, qEF, iE, qEG, al...","0.999, 0.042, 0.020, 0.014, 0.008, 0.007, 0.00..."
6,integrated_current,7,1.068485e+01,"kPyrO2, kPyrDrain, kAldO2, kPyrS_N, kAldPyr, k...","0.829, 0.431, 0.273, 0.161, 0.105, 0.065, 0.05..."
7,integrated_current,8,1.251908e+01,"betaF0, iG, qEF, iE, gammaG0, kAldO2, qEG, alp...","0.633, 0.632, 0.326, 0.224, 0.106, 0.091, 0.09..."


## DOE

In [9]:
pd.read_csv(RESULTS / 'candidate_ranking_integrated_selected.csv').head(15)

,candidate,family,medium,horizon_h,temperature_segments,N_pulses_kg_m3,candidate_logdet,candidate_min_eigenvalue,candidate_max_eigenvalue,candidate_min_relative_eigenvalue,...,var_ratio_qgas_threshold_fraction,var_reduction_qgas_threshold_fraction,target_mean_var_reduction,target_worst_var_reduction,aroma_co2_mean_var_reduction,aroma_co2_worst_var_reduction,hybrid_score,dopt_score,eopt_score,rationale
0,natural_pilot_cold_to_warm_earlyN,temperature_N,natural,300.0,"14, 16, 22, 20",30h:0.045,66.485730,1.675827e-10,18097.319095,9.260082e-15,...,0.806737,0.193263,0.281705,0.040737,0.186073,0.040737,95.826572,155.758730,-17.977814,Cold start followed by warm transition and ear...
1,natural_pilot_EA_cold_warm_Nsplit,EA_temperature_Nsplit,natural,300.0,"13, 18, 23, 19",32h:0.025; 80h:0.035,67.187325,9.096097e-11,16939.665689,5.369703e-15,...,0.694844,0.305156,0.268697,0.042529,0.198772,0.042529,95.760035,155.692015,-17.983539,Cold start then warm acceleration with split n...
2,natural_pilot_midN_temperature_step,temperature_N,natural,300.0,"16, 20, 23, 18",54h:0.04,65.011504,2.336080e-10,17290.355104,1.351089e-14,...,0.823494,0.176506,0.252715,0.037088,0.195191,0.084671,94.990167,155.372525,-18.193402,Temperature step with mid-growth nitrogen pert...
3,natural_pilot_low_temp_aroma_retention,aroma_retention,natural,300.0,"13, 15, 16, 16",54h:0.035,64.152376,5.126759e-10,19224.825387,2.666739e-14,...,0.770516,0.229484,0.272544,0.064638,0.181724,0.064638,94.956399,154.668940,-17.869067,Cold profile to contrast aroma retention again...
4,natural_pilot_EA_cold_retention_noN,EA_retention_reference,natural,300.0,"13, 13, 16, 18",NaN,59.312763,9.368424e-11,21895.012283,4.278794e-15,...,0.690030,0.309970,0.302438,0.072913,0.192651,0.072913,94.625655,154.926424,-18.139787,Low-temperature no-pulse comparator to decoupl...
5,natural_pilot_two_step_N_ladder,N_timing,natural,300.0,"17, 19, 21, 18",30h:0.03; 78h:0.035,60.845825,9.608032e-11,19703.212473,4.876379e-15,...,0.838795,0.161205,0.239714,0.029696,0.201249,0.091317,94.225049,154.978156,-18.383784,Two smaller nitrogen pulses to separate early ...
6,natural_pilot_high_rate_strip,co2_aroma,natural,300.0,"21, 24, 23, 19",30h:0.03,58.815102,8.952016e-12,18726.960292,4.780283e-16,...,0.889073,0.110927,0.295082,0.059407,0.206349,0.067724,94.127047,156.165669,-19.008050,High-rate natural fermentation to excite CO2 s...
7,natural_pilot_noN_dynamic_temperature,temperature,natural,300.0,"15, 22, 18, 22",NaN,62.742116,6.970868e-11,15067.867515,4.626313e-15,...,0.764065,0.235935,0.272291,0.060068,0.230348,0.125818,94.100093,155.099490,-18.528133,Temperature-only perturbation for settings whe...
8,natural_pilot_lateN_stationary_probe,N_timing,natural,300.0,"18, 20, 20, 18",80h:0.05,60.530219,1.083908e-10,15859.040237,6.834638e-15,...,0.821441,0.178559,0.238365,0.039964,0.206425,0.108860,93.481389,154.565436,-18.575951,Late nitrogen addition to test stationary/grow...
9,natural_pilot_EA_warm_early_noN,EA_temperature_Nstress,natural,300.0,"24, 23, 19, 17",NaN,58.380022,3.777556e-11,15650.267150,2.413733e-15,...,0.878205,0.121795,0.338981,0.049798,0.238857,0.057208,93.164408,156.615551,-19.759202,Warm early natural fermentation without nutrie...


In [10]:
pd.read_csv(RESULTS / 'selected_campaign_hybrid_integrated_selected.csv')

,objective,campaign_order,candidate,family,medium,horizon_h,temperature_segments,N_pulses_kg_m3,score,rationale,...,var_ratio_k_EA_XE,var_reduction_k_EA_XE,var_ratio_k_EA_XE_Nlim,var_reduction_k_EA_XE_Nlim,var_ratio_qgas_threshold_fraction,var_reduction_qgas_threshold_fraction,target_mean_var_reduction,target_worst_var_reduction,aroma_co2_mean_var_reduction,aroma_co2_worst_var_reduction
0,hybrid,1,natural_pilot_cold_to_warm_earlyN,temperature_N,natural,300.0,"14, 16, 22, 20",30h:0.045,95.826572,Cold start followed by warm transition and ear...,...,0.824426,0.175574,0.859193,0.140807,0.806737,0.193263,0.281705,0.040737,0.186073,0.040737
1,hybrid,2,natural_pilot_EA_warm_early_noN,EA_temperature_Nstress,natural,300.0,"24, 23, 19, 17",NaN,105.417412,Warm early natural fermentation without nutrie...,...,0.714023,0.285977,0.717129,0.282871,0.716362,0.283638,0.466309,0.125098,0.347603,0.197474
2,hybrid,3,natural_pilot_warm_to_cool_noN,temperature,natural,300.0,"22, 22, 17, 16",NaN,110.818974,"Warm early phase to excite growth and CO2, the...",...,0.631989,0.368011,0.629291,0.370709,0.666466,0.333534,0.531127,0.147985,0.429702,0.219544
3,hybrid,4,natural_pilot_EA_cold_warm_Nsplit,EA_temperature_Nsplit,natural,300.0,"13, 18, 23, 19",32h:0.025; 80h:0.035,115.408364,Cold start then warm acceleration with split n...,...,0.558425,0.441575,0.569878,0.430122,0.526713,0.473287,0.585935,0.184908,0.493700,0.301174
4,hybrid,5,natural_pilot_low_temp_aroma_retention,aroma_retention,natural,300.0,"13, 15, 16, 16",54h:0.035,119.087785,Cold profile to contrast aroma retention again...,...,0.503231,0.496769,0.524215,0.475785,0.466920,0.533080,0.624169,0.211503,0.532185,0.345771
5,hybrid,6,natural_pilot_EA_warm_early_lateN,EA_lateN,natural,300.0,"23, 22, 20, 18",98h:0.045,122.457244,Warm early fermentation followed by late nitro...,...,0.455449,0.544551,0.477437,0.522563,0.445633,0.554367,0.653524,0.226065,0.566778,0.354905
